# 01 – Contrôle qualité des données

Avant toute analyse, je vérifie si les données sont fiables. Je mesure seulement,
sans rien modifier : les corrections sont faites dans `02_nettoyage.ipynb`.

Je regroupe les contrôles en quatre familles : complétude, unicité, validité, cohérence.

In [1]:
import os
import duckdb
import pandas as pd


if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Dossier de travail :", os.getcwd())

con = duckdb.connect() 
con.execute("SET TimeZone = 'UTC'") 

Dossier de travail : c:\Users\USER\projets\doctoome-case-hamady


## 1. Vue d'ensemble

Je commence par les volumes de chaque table et la période réellement couverte.
Le README annonce juin–août 2026 : je vérifie les dates minimum et maximum.

In [2]:
query = """
    SELECT 'sessions' AS table_name,
           COUNT(*)                AS nb_lignes,
           MIN(session_started_at) AS date_min,
           MAX(session_started_at) AS date_max
    FROM 'data/sessions.parquet'
    UNION ALL
    SELECT 'events', 
           COUNT(*), 
           MIN(event_timestamp), 
           MAX(event_timestamp)
    FROM 'data/questionnaire_events.parquet'
    UNION ALL
    SELECT 'answers', 
           COUNT(*), 
           MIN(answered_at), 
           MAX(answered_at)
    FROM 'data/questionnaire_answers.parquet'
    UNION ALL
    SELECT 'outcomes', 
           COUNT(*), 
           MIN(completed_at), 
           MAX(completed_at)
    FROM 'data/questionnaire_outcomes.parquet'
    UNION ALL
    SELECT 'visitors', 
           COUNT(*), 
           MIN(first_seen_at), 
           MAX(first_seen_at)
    FROM 'data/visitors.parquet'
"""

con.sql(query).df()

,table_name,nb_lignes,date_min,date_max
0,sessions,26000,2026-06-01 00:01:59.429044+00:00,2026-08-02 16:44:38.140839+00:00
1,events,195816,2026-06-01 00:01:59.429044+00:00,2026-08-02 16:44:38.140839+00:00
2,answers,68460,2026-06-01 00:02:35.588484+00:00,2026-08-02 16:35:18.867974+00:00
3,outcomes,11357,2026-06-01 00:12:52.331208+00:00,2026-08-02 16:35:20.125069+00:00
4,visitors,20000,2026-06-01 00:01:59.429044+00:00,2026-08-02 16:44:38.140839+00:00


Les cinq tables couvrent la même période, du 1er juin au 2 août 2026 (UTC).
Le README annonce juin–août, mais les données s'arrêtent le 2 août : août ne représente
que 2 jours. Donc tout raisonnement par mois ou comparaison de mois serait trompeur.

## 2. Complétude

Je compte les valeurs manquantes dans les colonnes descriptives.

In [3]:
tables = {
    "visitors": "data/visitors.parquet",
    "sessions": "data/sessions.parquet",
    "events":   "data/questionnaire_events.parquet",
    "answers":  "data/questionnaire_answers.parquet",
    "outcomes": "data/questionnaire_outcomes.parquet",
}

lignes = []
for nom, chemin in tables.items():
    df = pd.read_parquet(chemin)
    manquants = df.isna().sum()
    for colonne, nb in manquants[manquants > 0].items():
        lignes.append({"table": nom, "colonne": colonne, "nb_manquants": nb,
                       "pct": round(100 * nb / len(df), 1)})

pd.DataFrame(lignes)

,table,colonne,nb_manquants,pct
0,visitors,birth_year,761,3.8
1,visitors,gender,653,3.3
2,visitors,region,804,4.0
3,sessions,campaign_name,14081,54.2
4,events,question_number,54131,27.6


Les manquants touchent peu de colonnes.Côté visiteurs, l'âge, le genre et la région
manquent pour 3 à 4 % des personnes : c'est faible, je les garderai sous la catégorie
« Non renseigné ».

In [4]:
con.sql("""
    SELECT event_type,
           COUNT(*)                         AS nb,
           COUNT(*) - COUNT(question_number) AS sans_question
    FROM 'data/questionnaire_events.parquet'
    GROUP BY event_type
    ORDER BY nb DESC
""").df()

,event_type,nb,sans_question
0,question_view,73225,0
1,question_answer,68460,0
2,landing_view,26000,26000
3,questionnaire_start,16774,16774
4,questionnaire_complete,11357,11357


Les 27,6 % de `question_number` vides dans les événements sont normaaux car des evenements types `landing_view`, `questionnaire_start`, `questionnaire_complete` ne se rapportent pas à une question spécifique.

In [5]:
query = """
    SELECT acquisition_source,
           COUNT(*)                        AS nb_sessions,
           COUNT(*) - COUNT(campaign_name) AS sans_campagne
    FROM 'data/sessions.parquet'
    GROUP BY acquisition_source
    ORDER BY nb_sessions DESC
"""
con.sql(query).df()

,acquisition_source,nb_sessions,sans_campagne
0,google_organic,6590,6590
1,google_ads,6375,0
2,social,3176,0
3,chatgpt,2401,2401
4,referral,1790,1790
5,email,1707,0
6,direct,1617,1398
7,other,674,674
8,google,450,450
9,Google,450,450


La campagne est absente pour les sources naturelles (Google organique, ChatGPT, referral)
et toujours renseignée pour les sources payantes ou gérées par Doctoome (Google Ads,
réseaux sociaux, e-mail) : ces 54 % de vides sont donc attendus. 

Seule exception : 219 sessions « direct » portent une campagne, ce qui suggère une attribution perdue. 

Je note aussi des libellés en double comme Google/google, traités dans la partie validité.

## 3. Unicité

Je vérifie deux choses : que chaque identifiant est unique dans sa table, puis qu'une même
action n'a pas été enregistrée plusieurs fois (doublons).

In [6]:
query = """
    SELECT 'visitors' AS table_name, 
           COUNT(*) AS nb_lignes, 
           COUNT(DISTINCT visitor_id) AS nb_ids_distincts
    FROM 'data/visitors.parquet'
    UNION ALL SELECT 'sessions', 
           COUNT(*), 
           COUNT(DISTINCT session_id) 
    FROM 'data/sessions.parquet'
    UNION ALL SELECT 'events',   
           COUNT(*), 
           COUNT(DISTINCT event_id)   
    FROM 'data/questionnaire_events.parquet'
    UNION ALL SELECT 'answers',  
           COUNT(*), 
           COUNT(DISTINCT answer_id)  
    FROM 'data/questionnaire_answers.parquet'
    UNION ALL SELECT 'outcomes', 
           COUNT(*), 
           COUNT(DISTINCT session_id) 
    FROM 'data/questionnaire_outcomes.parquet'
"""

con.sql(query).df()

,table_name,nb_lignes,nb_ids_distincts
0,visitors,20000,20000
1,sessions,26000,26000
2,events,195816,195816
3,answers,68460,68460
4,outcomes,11357,11357


Aucun identifiant n'est en double.

In [7]:
query = """
    WITH groupes AS (
        SELECT session_id, question_number,
               COUNT(*)                     AS nb_reponses,
               COUNT(DISTINCT answer_value) AS nb_valeurs_differentes
        FROM 'data/questionnaire_answers.parquet'
        GROUP BY session_id, question_number
        HAVING COUNT(*) > 1
    )
    SELECT COUNT(*)                                                    AS questions_repondues_plusieurs_fois,
           SUM(CASE WHEN nb_valeurs_differentes > 1 THEN 1 ELSE 0 END) AS avec_valeurs_differentes,
           SUM(nb_reponses - 1)                                        AS lignes_en_trop
    FROM groupes
"""

con.sql(query).df()

,questions_repondues_plusieurs_fois,avec_valeurs_differentes,lignes_en_trop
0,697,697.0,697.0


697 questions ont reçu deux réponses dans la même session, toujours avec deux valeurs différentes

In [8]:
con.sql("""
    WITH groupes AS (
        SELECT session_id, event_type, question_number, COUNT(*) AS nb
        FROM 'data/questionnaire_events.parquet'
        GROUP BY session_id, event_type, question_number
        HAVING COUNT(*) > 1
    )
    SELECT event_type,
           COUNT(*)    AS groupes_en_double,
           SUM(nb - 1) AS lignes_en_trop
    FROM groupes
    GROUP BY event_type
    ORDER BY lignes_en_trop DESC
""").df()

,event_type,groupes_en_double,lignes_en_trop
0,question_view,1043,1043.0
1,question_answer,697,697.0


1043 + 697 sont des events de type `question_view` et `question_answer` qui sont en double. 

Au nettoyage  dans `answers`, je garde la dernière réponse à chaque question  ; dans `events`, la dernière réponse et la première vue de chaque question.

## 4. Validité
Je vérifie que les valeurs sont écrites de façon homogène et restent dans les valeurs attendues.

In [9]:
query = """
    SELECT 'gender' AS colonne, gender AS valeur, COUNT(*) AS nb FROM 'data/visitors.parquet' GROUP BY gender
    UNION ALL
    SELECT 'region', region, COUNT(*) FROM 'data/visitors.parquet' GROUP BY region
    ORDER BY colonne, valeur
"""


con.sql(query).df()

,colonne,valeur,nb
0,gender,F,283
1,gender,Female,219
2,gender,M,245
3,gender,Male,246
4,gender,female,9301
5,gender,male,8257
6,gender,other,284
7,gender,prefer_not_to_say,512
8,gender,NaN,653
9,region,Auvergne Rhone Alpes,103


Une même catégorie est écrite de plusieurs façons comme `female`/`F`/`Female`, `bretagne`/`Bretagne`, ...

In [10]:
query = """
    SELECT MIN(birth_year) AS plus_ancienne, MAX(birth_year) AS plus_recente,
           SUM(CASE WHEN 2026 - birth_year < 18 THEN 1 ELSE 0 END) AS mineurs
    FROM 'data/visitors.parquet'
"""

con.sql(query).df()

,plus_ancienne,plus_recente,mineurs
0,1929,2010,42.0


Les années vont de 1929 à 2010 : pas de valeur aberrante, mais 42 visiteurs ont moins de 18 ans.

In [11]:
query = """
    WITH autorisees AS (
        SELECT question_number,
               UNNEST(CAST(json(allowed_answers) AS VARCHAR[])) AS valeur
        FROM 'data/questionnaire_questions.parquet'
    )
    SELECT a.question_number, a.answer_value, COUNT(*) AS nb
    FROM 'data/questionnaire_answers.parquet' a
    LEFT JOIN autorisees q
           ON a.question_number = q.question_number AND a.answer_value = q.valeur
    WHERE q.valeur IS NULL
    GROUP BY ALL
"""
con.sql(query).df()

,question_number,answer_value,nb


Toutes les réponses font partie des valeurs autorisées pour leur question.

## 5. Cohérence
Je vérifie que les tables racontent la même histoire.

*Lignes orphelines et fins sans résultat*

In [12]:
query = """
    SELECT
      (SELECT COUNT(*) FROM 'data/sessions.parquet'
         WHERE visitor_id NOT IN (SELECT visitor_id FROM 'data/visitors.parquet')) AS sessions_sans_visiteur,
      (SELECT COUNT(*) FROM 'data/questionnaire_answers.parquet'
         WHERE session_id NOT IN (SELECT session_id FROM 'data/sessions.parquet')) AS reponses_sans_session,
      (SELECT COUNT(DISTINCT session_id) FROM 'data/questionnaire_events.parquet'
         WHERE event_type = 'questionnaire_complete'
           AND session_id NOT IN (SELECT session_id FROM 'data/questionnaire_outcomes.parquet')) AS fins_sans_resultat
"""

con.sql(query).df()

,sessions_sans_visiteur,reponses_sans_session,fins_sans_resultat
0,0,0,0


In [13]:
query = """
    WITH reponses AS (
        SELECT session_id, COUNT(DISTINCT question_number) AS nb_questions
        FROM 'data/questionnaire_answers.parquet'
        GROUP BY session_id
    ),
    fins AS (
        SELECT DISTINCT session_id
        FROM 'data/questionnaire_events.parquet'
        WHERE event_type = 'questionnaire_complete'
    ),
    resultats AS (
        SELECT session_id FROM 'data/questionnaire_outcomes.parquet'
    )
    SELECT
      (SELECT COUNT(*) FROM fins)                                       AS sessions_terminees,
      (SELECT COUNT(*) FROM resultats
         WHERE session_id NOT IN (SELECT session_id FROM fins))         AS resultats_sans_fin,
      (SELECT COUNT(*) FROM fins f JOIN reponses r USING (session_id)
         WHERE r.nb_questions = 5)                                      AS terminees_avec_5_reponses,
      (SELECT COUNT(*) FROM reponses r
         WHERE r.nb_questions = 5
           AND session_id NOT IN (SELECT session_id FROM fins)
           AND session_id NOT IN (SELECT session_id FROM resultats))    AS _5_reponses_sans_fin_ni_resultat
"""
con.sql(query).df()

,sessions_terminees,resultats_sans_fin,terminees_avec_5_reponses,_5_reponses_sans_fin_ni_resultat
0,11357,0,11357,105


Aucune ligne orpheline, et chaque questionnaire terminé (`questionnaire_complete`) a un résultat (`outcome`) et vice-versa. Mais 105 sessions ont répondu aux 5 questions sans fin ni résultat : à surveiller.

In [14]:
query = """
    WITH q1 AS (
        SELECT session_id, answer_value
        FROM 'data/questionnaire_answers.parquet'
        WHERE question_number = 1
        QUALIFY ROW_NUMBER() OVER (PARTITION BY session_id ORDER BY answered_at DESC) = 1
    )
    SELECT q1.answer_value AS reponse_q1, o.outcome_category, COUNT(*) AS nb
    FROM 'data/questionnaire_outcomes.parquet' o
    JOIN q1 USING (session_id)
    GROUP BY ALL
    ORDER BY reponse_q1, outcome_category
"""

con.sql(query).df()

,reponse_q1,outcome_category,nb
0,no,no_current_indication,7508
1,no,possible_risk,2152
2,prefer_not_to_say,no_current_indication,333
3,prefer_not_to_say,possible_risk,124
4,yes,declared_diagnosed,1240


En prenant la dernière réponse à la Q1, la catégorie finale est toujours cohérente : « yes »
donne toujours `declared_diagnosed`

In [15]:
query = """
    WITH reponses AS (
        SELECT session_id,
               COUNT(DISTINCT question_number) AS nb_questions,
               MAX(question_number)            AS derniere_question
        FROM 'data/questionnaire_answers.parquet'
        GROUP BY session_id
    )
    SELECT COUNT(*) AS sessions_avec_reponses,
           SUM(CASE WHEN nb_questions <> derniere_question THEN 1 ELSE 0 END) AS questions_sautees
    FROM reponses
"""

con.sql(query).df()

,sessions_avec_reponses,questions_sautees
0,15558,0.0


Tous les parcours sont logiques, je veux dire qu'il n'y a eu de sauts par exemple répondre Q3 sans répondre Q1 ou Q2.

## 6. Récapitulatif

| Contrôle | Constat | Décision au nettoyage |
|---|---|---|
| Période | 1er juin – 2 août 2026 | Analyse par semaine |
| Complétude | 3–4 % de démographie manquante ; campagnes vides attendues | Catégorie « Non renseigné » |
| Unicité | 697 réponses modifiées, 1 043 vues en double | Dernière réponse, première vue |
| Validité | Libellés incohérents (genre, région, source) ; 42 répondants, mineurs, entre 16-17 ans | Libellés unifiés |
| Cohérence | Pas d'orphelin ; tout est cohérent sauf les 105 sessions à 5 réponses sans résultat | Comptées comme non terminées |